## Overfitting and Underfitting Analysis

Validation-stage Overfitting / Underfitting Check

During model development, the validation set is used to examine whether a model generalises well to unseen data. Instead of relying on the test set, the gap between training error and validation error provides an indication of potential overfitting or underfitting.

If the validation error is significantly higher than the training error, the model may be overfitting the training data. Conversely, if both errors remain relatively high, the model may not have learned sufficient patterns from the data.

Baseline vs Enhanced Model

Two neural network models are evaluated:

1. Model 1 (Baseline): uses the original feature set.

2. Model 2 (Enhanced): adds an additional spatial feature representing the mean price of neighbouring properties, derived using K-nearest neighbours based on geographical coordinates.

This additional feature is intended to capture local price patterns that may not be fully represented by the original attributes.

In [ ]:
# ============================================================
# Validation-stage Overfitting / Underfitting Check
# 验证阶段过拟合 / 欠拟合检查
#
# Purpose:
# Use the train-validation gap (rather than the test set) to assess
# whether each model shows signs of overfitting or underfitting.
# 目的：
# 通过比较训练误差与验证误差（而不是使用测试集），
# 判断模型是否出现过拟合或欠拟合。
#
# Why validation instead of test:
# The validation set is the appropriate reference during model
# development and model comparison.
# The test set should mainly be reserved for final performance reporting.
# 为什么使用validation而不是test：
# validation用于模型开发和模型选择阶段，
# test集主要用于最终模型性能报告。
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from sklearn.neighbors import NearestNeighbors

# ------------------------------------------------------------
# 1. Validation-stage baseline model
# 验证阶段：基准模型
# ------------------------------------------------------------

scaler_base_valcheck = StandardScaler()
# Create a scaler to standardise features
# 创建标准化器用于特征标准化

X_train_base_valcheck = scaler_base_valcheck.fit_transform(X_train_raw)
# Fit scaler on training data and transform training features
# 用训练数据拟合scaler并转换训练特征

X_val_base_valcheck   = scaler_base_valcheck.transform(X_val_raw)
# Apply the same transformation to validation data
# 使用同样的标准化参数转换验证数据

baseline_valcheck_model = MLPRegressor(
    hidden_layer_sizes=(128, 32),  # neural network structure / 神经网络结构
    activation='tanh',             # activation function / 激活函数
    solver='adam',                 # optimiser / 优化算法
    learning_rate_init=1e-3,       # learning rate / 学习率
    max_iter=800,                  # maximum training iterations / 最大训练轮数
    early_stopping=False,          # disable internal early stopping / 不启用早停
    random_state=26                # ensure reproducibility / 保证结果可复现
)

baseline_valcheck_model.fit(X_train_base_valcheck, y_train)
# Train the baseline neural network
# 训练基准神经网络模型

baseline_train_pred_valcheck = baseline_valcheck_model.predict(X_train_base_valcheck)
# Predictions on training data
# 对训练数据进行预测

baseline_val_pred_valcheck   = baseline_valcheck_model.predict(X_val_base_valcheck)
# Predictions on validation data
# 对验证数据进行预测

m1_train_mse_val = mean_squared_error(y_train, baseline_train_pred_valcheck)
# Training Mean Squared Error
# 训练集均方误差

m1_val_mse       = mean_squared_error(y_val, baseline_val_pred_valcheck)
# Validation Mean Squared Error
# 验证集均方误差

# training loss curve (approximate MSE)
# 训练过程loss曲线（近似MSE）
train_loss_curve_m1_val = np.array(baseline_valcheck_model.loss_curve_) * 2


# ------------------------------------------------------------
# 2. Validation-stage enhanced model
# 验证阶段：增强模型（加入邻域价格特征）
# ------------------------------------------------------------

best_K_valcheck = best_K
# Use previously selected optimal K for neighbour search
# 使用之前确定的最佳邻居数量K

knn_valcheck = NearestNeighbors(n_neighbors=best_K_valcheck + 1)
# Create KNN model to find spatial neighbours
# 创建KNN模型用于寻找空间邻居

knn_valcheck.fit(X_train_raw[['Latitude', 'Longitude']])
# Fit KNN using training coordinates
# 使用训练数据的经纬度训练KNN

y_train_aligned_valcheck = y_train.loc[X_train_raw.index].to_numpy()
# Align training target values with feature indices
# 确保训练标签与特征索引对齐

# training neighbours (remove self)
# 查找训练样本邻居（去掉自身）
_, idx_train_valcheck = knn_valcheck.kneighbors(X_train_raw[['Latitude', 'Longitude']])
idx_train_valcheck = idx_train_valcheck[:, 1:best_K_valcheck + 1]

# validation neighbours (query against training only)
# 查找验证样本邻居（只在训练集中查询）
_, idx_val_valcheck = knn_valcheck.kneighbors(X_val_raw[['Latitude', 'Longitude']])
idx_val_valcheck = idx_val_valcheck[:, :best_K_valcheck]

train_prices_valcheck = y_train_aligned_valcheck[idx_train_valcheck]
# Prices of neighbouring houses for training samples
# 训练样本邻居房价

val_prices_valcheck   = y_train_aligned_valcheck[idx_val_valcheck]
# Prices of neighbouring houses for validation samples
# 验证样本邻居房价

# neighbour mean price feature
# 邻居平均房价特征
neigh_mean_train_valcheck = train_prices_valcheck.mean(axis=1)
neigh_mean_valcheck       = val_prices_valcheck.mean(axis=1)

X_train_enh_valcheck_raw = X_train_raw.copy()
X_val_enh_valcheck_raw   = X_val_raw.copy()

X_train_enh_valcheck_raw["NeighbourPriceMean"] = neigh_mean_train_valcheck
# Add neighbour price feature to training set
# 向训练集添加邻居房价特征

X_val_enh_valcheck_raw["NeighbourPriceMean"]   = neigh_mean_valcheck
# Add neighbour price feature to validation set
# 向验证集添加邻居房价特征

scaler_enh_valcheck = StandardScaler()
# Standardise enhanced feature set
# 对增强后的特征再次进行标准化

X_train_enh_valcheck = scaler_enh_valcheck.fit_transform(X_train_enh_valcheck_raw)
X_val_enh_valcheck   = scaler_enh_valcheck.transform(X_val_enh_valcheck_raw)

enhanced_valcheck_model = MLPRegressor(
    hidden_layer_sizes=(128, 32),
    activation='tanh',
    solver='adam',
    learning_rate_init=1e-3,
    max_iter=800,
    early_stopping=False,
    random_state=26
)

enhanced_valcheck_model.fit(X_train_enh_valcheck, y_train)
# Train enhanced model
# 训练增强模型

enh_train_pred_valcheck = enhanced_valcheck_model.predict(X_train_enh_valcheck)
enh_val_pred_valcheck   = enhanced_valcheck_model.predict(X_val_enh_valcheck)

m2_train_mse_val = mean_squared_error(y_train, enh_train_pred_valcheck)
m2_val_mse       = mean_squared_error(y_val, enh_val_pred_valcheck)

# training loss curve (approximate MSE)
# 记录训练损失曲线
train_loss_curve_m2_val = np.array(enhanced_valcheck_model.loss_curve_) * 2

Validation/Train Error Ratio

To provide a simple diagnostic, the ratio between validation MSE and training MSE is calculated.

1. A large ratio suggests potential overfitting.

2. A ratio close to 1 indicates a small generalisation gap.

3. If both errors remain high despite a small gap, the model may still be underfitting.

This ratio offers a quick numerical indicator of generalisation performance during the validation stage.

In [ ]:
# ------------------------------------------------------------
# 3. Simple diagnosis based on validation / train ratio
# 基于验证误差 / 训练误差比例的简单诊断
# ------------------------------------------------------------

def diagnose_validation(model_name, train_mse, val_mse, threshold_ratio=1.5):
    # Function to diagnose overfitting or underfitting
    # 根据误差比例判断过拟合或欠拟合

    ratio = val_mse / train_mse if train_mse > 0 else float('inf')
    # Compute validation/train error ratio
    # 计算验证误差与训练误差的比例

    print(f"\n{'='*60}")
    print(model_name)
    print(f"{'='*60}")
    print(f"Train MSE             : {train_mse:.4f}")
    print(f"Validation MSE        : {val_mse:.4f}")
    print(f"Validation/Train Ratio: {ratio:.3f}")

    if ratio > threshold_ratio:
        print("Diagnosis: possible overfitting (validation error notably exceeds training error)")
    elif ratio < 1.1:
        print("Diagnosis: small generalisation gap — possible underfitting if both errors remain relatively high compared with earlier configurations")
    else:
        print("Diagnosis: moderate generalisation gap / reasonable fit")

    print(f"{'='*60}")


diagnose_validation("Model 1 — Baseline (validation stage)", m1_train_mse_val, m1_val_mse)
# Run diagnosis for baseline model
# 对基准模型进行诊断

diagnose_validation("Model 2 — Enhanced (validation stage)", m2_train_mse_val, m2_val_mse)
# Run diagnosis for enhanced model
# 对增强模型进行诊断


Model Performance Visualisation

Two visualisations are used to interpret model behaviour:

1. Train vs Validation MSE Bar Chart
  This chart directly compares the training and validation errors of the two models.

2. Learning Curves
  The learning curves show how the training loss evolves across epochs.
  The final training and validation MSE values are also plotted as reference lines to help assess the generalisation gap.

Together, these visualisations provide an intuitive view of the models’ fitting behaviour and stability during training.

In [ ]:
# ------------------------------------------------------------
# 4. Visualisation 1 — Train vs Validation MSE
# 可视化1：训练误差 vs 验证误差
# ------------------------------------------------------------

labels = ['Model 1\n(Baseline)', 'Model 2\n(Enhanced)']
# Labels for models
# 模型标签

train_mses_val = [m1_train_mse_val, m2_train_mse_val]
# Training errors
# 训练误差

val_mses       = [m1_val_mse, m2_val_mse]
# Validation errors
# 验证误差

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
# Create bar chart
# 创建柱状图

bars1 = ax.bar(x - width/2, train_mses_val, width, label='Train MSE', color='steelblue', alpha=0.85)
bars2 = ax.bar(x + width/2, val_mses, width, label='Validation MSE', color='darkorange', alpha=0.85)

ax.set_ylabel('MSE')
ax.set_title('Train vs Validation MSE — Model 1 and Model 2')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

for bar, val in zip(bars1, train_mses_val):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
# Display values on training bars
# 在训练柱状图上显示数值

for bar, val in zip(bars2, val_mses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
# Display values on validation bars
# 在验证柱状图上显示数值

plt.tight_layout()
plt.show()


# ------------------------------------------------------------
# 5. Visualisation 2 — Learning Curves
# 可视化2：学习曲线
# ------------------------------------------------------------

print(f"Model 1 actual training epochs: {len(train_loss_curve_m1_val)}")
print(f"Model 2 actual training epochs: {len(train_loss_curve_m2_val)}")
print("Note: fewer epochs than max_iter indicate optimiser convergence before max_iter.")
# Print actual number of training epochs
# 打印实际训练轮数

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

global_min = min(
    train_loss_curve_m1_val.min(),
    train_loss_curve_m2_val.min(),
    m1_val_mse,
    m2_val_mse
)

global_max = max(
    train_loss_curve_m1_val.max(),
    train_loss_curve_m2_val.max(),
    m1_val_mse,
    m2_val_mse
)
# Set unified y-axis scale
# 统一y轴范围

for ax, lc_tr, final_train_mse, final_val_mse, title in zip(
    axes,
    [train_loss_curve_m1_val, train_loss_curve_m2_val],
    [m1_train_mse_val, m2_train_mse_val],
    [m1_val_mse, m2_val_mse],
    ['Model 1 — Baseline (validation stage)', 'Model 2 — Enhanced (validation stage)']
):

    epochs = range(1, len(lc_tr) + 1)

    ax.plot(epochs, lc_tr, label='Train MSE', color='steelblue')
    # Plot training learning curve
    # 绘制训练误差曲线

    ax.axhline(
        y=final_train_mse,
        color='steelblue',
        linestyle=':',
        linewidth=1.2,
        label=f'Final Train MSE = {final_train_mse:.4f}'
    )

    ax.axhline(
        y=final_val_mse,
        color='darkorange',
        linestyle='--',
        linewidth=1.2,
        label=f'Final Validation MSE = {final_val_mse:.4f}'
    )
    # Horizontal lines showing final errors
    # 水平线表示最终训练误差与验证误差

    ax.set_ylim(global_min * 0.9, global_max * 1.05)
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE')
    ax.legend(fontsize=8)
    ax.grid(True, linestyle=':', alpha=0.6)

plt.suptitle('Learning Curves — Validation-stage Generalisation Check', fontsize=13)
plt.tight_layout()
plt.show()

## Conclusion

The validation-stage analysis suggests that both models achieve reasonable generalisation performance, with validation errors remaining close to their corresponding training errors. This indicates that the neural networks do not exhibit strong signs of overfitting on the training data.

Comparing the two approaches, the enhanced model that incorporates the neighbourhood price feature provides a slight improvement in predictive performance. This suggests that spatial information captured through neighbouring property prices contributes useful contextual information for the prediction task.

Overall, the validation results support the effectiveness of the enhanced feature design, while the learning curves indicate stable optimisation behaviour during training.